In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
# Initialize Spark with Kafka package
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3"
    ) \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1eed4735-1a29-4436-9f65-e6c1c4604fdb;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-

In [3]:
# 1. Load Static User Data (CSV)

users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True
)

In [ ]:
# 2. Read Streaming Data from Kafka

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])
# Kafka stores messages as JSON strings.Spark needs to know : Which fields exist ,Their data types... That's why we need a schema.

kafka_stream = spark.readStream \
    .format("kafka") \             # Kafka → Spark
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

In [5]:
# 3. Parse and filter data
parsed_stream = kafka_stream.select(
    from_json(col("value").cast("string"), tx_schema).alias("tx")
).select("tx.*")

fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [6]:
# 4. Enrich Stream with User Details
enriched_fraud = fraud_stream.join(users_df, "userId")

In [7]:
# 5. Format for Output Kafka Topic
output_stream = enriched_fraud \
    .withColumn(
        "value",
        to_json(struct("*")).cast("string")
    ) \
    .select("value")

In [ ]:
# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option("checkpointLocation", "/workspace/checkpoints") \
    .start()
query.awaitTermination()

26/06/18 11:28:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/18 11:28:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/06/18 11:28:59 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                